
### Notebook 12 — Real-Data Experiment (B): financial returns
  - "Do LCT-B and Fisher-z DIVERGE on heavy-tailed real data?"
  - Counterpart to notebook 11 (ABIDE fMRI, near-Gaussian, methods AGREE).
  - Here returns are heavy-tailed (kappa >> 1), so Fisher-z should over-reject
  - while LCT-B holds -- the divergence the simulations predict, in the wild.

In [1]:
 # run once if needed
# !pip install yfinance    

In [2]:
# Cell 1: install + imports + config
import sys, numpy as np, pandas as pd
import yfinance as yf
sys.path.insert(0, "..")   # so we can import src/ (assumes this file is in notebooks/)

# Two regimes framed as the two-sample groups (H0: rho_ij,1 == rho_ij,2):
#   Group 1 = calm    (2019)  -> lower cross-asset correlation, moderate tails
#   Group 2 = volatile (2020) -> correlations spike in the crash, very heavy tails
P1_START, P1_END = "2019-01-01", "2019-12-31"   # calm
P2_START, P2_END = "2020-01-01", "2020-12-31"   # volatile (incl. COVID crash)

ALPHA = 0.05
B_BOOT = 100          # bootstrap replicates for LCT-B (paper default at this scale)
RNG_SEED = 0

# ~205 large-cap U.S. equities (broad sector spread, all trading through 2019-2020).
# Any delisted/renamed tickers that return no data are dropped automatically in Cell 2,
# so a handful of stragglers is harmless -- we just want ~200 clean names.
TICKERS = [
    # Information technology
    "AAPL","MSFT","NVDA","AVGO","ORCL","CRM","ADBE","CSCO","ACN","TXN","QCOM","INTC","AMD",
    "IBM","INTU","NOW","AMAT","MU","ADI","LRCX","KLAC","SNPS","CDNS","NXPI","MCHP","FTNT",
    "ANET","ROP","APH","MSI","GLW","HPQ","KEYS",
    # Communication services
    "GOOGL","GOOG","META","DIS","CMCSA","NFLX","T","VZ","TMUS","CHTR","EA","OMC","IPG",
    # Consumer discretionary
    "AMZN","HD","MCD","NKE","LOW","SBUX","TJX","BKNG","TGT","GM","F","MAR","ROST","YUM",
    "ORLY","AZO","DHI","LEN","CMG","APTV","BBY","EBAY","HLT","GPC","POOL","TSCO",
    # Consumer staples
    "PG","KO","PEP","WMT","COST","PM","MO","MDLZ","CL","KMB","GIS","SYY","KHC","HSY","STZ",
    "KR","CLX","MKC","ADM","K","HRL","CAG","TAP",
    # Health care
    "UNH","JNJ","LLY","ABBV","MRK","PFE","TMO","ABT","DHR","BMY","AMGN","CVS","MDT","ISRG",
    "GILD","SYK","VRTX","REGN","ZTS","BSX","BDX","HCA","CI","HUM","CNC","BIIB","IQV","IDXX",
    "A","DXCM","EW","BAX","RMD","MTD",
    # Financials
    "BRK-B","JPM","BAC","WFC","GS","MS","C","SCHW","AXP","BLK","SPGI","CB","MMC","PGR","PNC",
    "USB","TFC","COF","BK","AIG","MET","AON","ICE","CME","AFL","ALL","TRV","PRU","STT","MCO",
    # Industrials
    "HON","UNP","UPS","BA","CAT","GE","LMT","RTX","DE","MMM","EMR","ETN","ITW","CSX","NSC",
    "FDX","GD","NOC","WM","PH","CARR","OTIS","PCAR","CTAS","JCI","CMI","GWW","FAST","AME",
    # Energy
    "XOM","CVX","COP","SLB","EOG","MPC","PSX","VLO","OXY","WMB","KMI","HAL","DVN","HES",
    # Materials
    "LIN","APD","SHW","ECL","FCX","NEM","DOW","DD","NUE","PPG","VMC","MLM","IFF",
    # Real estate
    "AMT","PLD","CCI","EQIX","PSA","O","SPG","WELL","DLR","AVB","EQR",
    # Utilities
    "NEE","DUK","SO","D","AEP","EXC","SRE","XEL","ED","PEG","WEC","ES","AEE","DTE",
]
TICKERS = sorted(set(TICKERS))

# Tickers Yahoo currently 404s (delisted/renamed/merged since 2020). They were being
# dropped anyway, so removing them up front just cleans the output -- p is unchanged.
_YF_DEAD = {"MMC", "HES", "K", "BK", "IPG"}
TICKERS = sorted(set(TICKERS) - _YF_DEAD)

# Quiet yfinance's logger so any stray per-ticker warnings don't bury real errors.
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

print(f"requested tickers: {len(TICKERS)}")

requested tickers: 235


In [3]:
# Cell 2: download + build the two return matrices
# Download with a BUFFER before P1_START so the first in-period return is well
# defined (otherwise the first trading day of 2019 is NaN for every stock, and the
# per-period dropna below would wipe out every column -> p = 0).

DL_START = "2018-12-10"
raw = yf.download(TICKERS, start=DL_START, end="2021-01-01",
                  auto_adjust=True, progress=False)["Close"]

# Drop tickers that failed to download entirely (all-NaN columns, e.g. delisted names)
raw = raw.dropna(axis=1, how="all")
print(f"tickers with data: {raw.shape[1]}")

# Daily log returns (first row overall is NaN, but it falls in the Dec-2018 buffer)
logret = np.log(raw / raw.shift(1))

def period_matrix(logret_df, start, end):
    sub = logret_df.loc[start:end]
    # keep only tickers with complete data IN THIS window (no spurious leading NaN now)
    sub = sub.dropna(axis=1, how="any")
    return sub

r1_df = period_matrix(logret, P1_START, P1_END)
r2_df = period_matrix(logret, P2_START, P2_END)

# Restrict to tickers present in BOTH periods, in the same column order
common = sorted(set(r1_df.columns) & set(r2_df.columns))
X = r1_df[common].to_numpy()     # (n1 days x p) -- calm
Y = r2_df[common].to_numpy()     # (n2 days x p) -- volatile
n1, p = X.shape
n2 = Y.shape[0]
M = p * (p - 1) // 2

print(f"p (stocks kept)      : {p}")
print(f"n1 (calm 2019 days)  : {n1}")
print(f"n2 (volatile 2020)   : {n2}")
print(f"candidate edges M    : {M}")

tickers with data: 235
p (stocks kept)      : 232
n1 (calm 2019 days)  : 252
n2 (volatile 2020)   : 253
candidate edges M    : 26796


In [4]:
# Cell 3: measure kappa (the bridge to the simulations)
from src.LCT import _kappa_hat, _zscore_columns

def scalar_kappa(A):
    return _kappa_hat(_zscore_columns(A))

# Also report plain excess kurtosis (mean over columns) for intuition:
def mean_excess_kurt(A):
    Az = _zscore_columns(A)
    m4 = (Az**4).mean(axis=0)          # standardized 4th moment per column
    return float(np.mean(m4 - 3.0))    # excess kurtosis, 0 for Gaussian

print(f"kappa_hat  calm (2019)     : {scalar_kappa(X):.3f}   (Gaussian = 1.0)")
print(f"kappa_hat  volatile (2020) : {scalar_kappa(Y):.3f}")
print(f"mean excess kurtosis calm     : {mean_excess_kurt(X):.2f}   (Gaussian = 0)")
print(f"mean excess kurtosis volatile : {mean_excess_kurt(Y):.2f}")

kappa_hat  calm (2019)     : 3.033   (Gaussian = 1.0)
kappa_hat  volatile (2020) : 3.437
mean excess kurtosis calm     : 6.03   (Gaussian = 0)
mean excess kurtosis volatile : 7.23


In [5]:
# Cell 4: run LCT + Fisher-z on the two-sample comparison
from src.LCT import lct_edge_stat, lct_threshold_normal
try:
    from src.LCTB_v2 import lct_threshold_bootstrap
except ImportError:
    from src.LCTB import lct_threshold_bootstrap
from src.FisherBaselines import two_group_z_stat, pvals_from_Z, bh_threshold, by_threshold

iu, ju = np.triu_indices(p, 1)

def to_flat(mask):
    """Return a boolean vector over the M upper-triangle edges."""
    m = np.asarray(mask)
    return m[iu, ju] if m.ndim == 2 else m

# --- LCT (statistic + normal and bootstrap thresholds) ---
T, R1, R2 = lct_edge_stat(X, Y, var_method="cai_liu")
_, mask_lctn = lct_threshold_normal(T, alpha=ALPHA)
_, mask_lctb, _ = lct_threshold_bootstrap(X, Y, alpha=ALPHA, B=B_BOOT, rng=RNG_SEED)

# --- Fisher-z + BH / BY (same R1,R2 and the honest per-period n) ---
Z  = two_group_z_stat(R1, R2, n1, n2)
pv = pvals_from_Z(Z)[iu, ju]
sel_bh = bh_threshold(pv, ALPHA)
sel_by = by_threshold(pv, ALPHA)

def ndisc(mask):
    return int(to_flat(mask).sum())

print(f"Discoveries at alpha={ALPHA} (of {M} edges):")
print(f"  Fisher-z + BH : {ndisc(sel_bh)}")
print(f"  Fisher-z + BY : {ndisc(sel_by)}")
print(f"  LCT-N         : {ndisc(mask_lctn)}")
print(f"  LCT-B         : {ndisc(mask_lctb)}")

Discoveries at alpha=0.05 (of 26796 edges):
  Fisher-z + BH : 21954
  Fisher-z + BY : 18818
  LCT-N         : 16854
  LCT-B         : 13844


In [6]:
# Cell 5: agreement between methods (contrast with fMRI's clean nesting)
lctb_set = set(np.where(to_flat(mask_lctb))[0])
bh_set   = set(np.where(to_flat(sel_bh))[0])
inter = len(lctb_set & bh_set)
union = len(lctb_set | bh_set)

print(f"LCT-B discoveries      : {len(lctb_set)}")
print(f"BH discoveries         : {len(bh_set)}")
print(f"overlap (Jaccard)      : {inter/max(union,1):.3f}")
print(f"edges only LCT-B finds : {len(lctb_set - bh_set)}")
print(f"edges only BH finds    : {len(bh_set - lctb_set)}")

LCT-B discoveries      : 13844
BH discoveries         : 21954
overlap (Jaccard)      : 0.631
edges only LCT-B finds : 0
edges only BH finds    : 8110


In [7]:
# Cell 6: subsampling stability (the FDR proxy when truth is unknown)
# Resample days within each period, re-run, and measure how consistent each method's
# discovery set is across resamples. Unstable discoveries are the signature of
# over-rejection. Expected: LCT-B stable, Fisher-z + BH unstable.

K = 8                 # number of resamples (bump if you want tighter estimates)
FRAC = 0.7            # fraction of days kept per resample
rng = np.random.default_rng(RNG_SEED)

def discoveries_on(Xs, Ys, method):
    Ts, RA, RB = lct_edge_stat(Xs, Ys, var_method="cai_liu")
    if method == "LCT-B":
        _, mk, _ = lct_threshold_bootstrap(Xs, Ys, alpha=ALPHA, B=B_BOOT, rng=0)
        return set(np.where(to_flat(mk))[0])
    else:  # BH
        Zs = two_group_z_stat(RA, RB, Xs.shape[0], Ys.shape[0])
        pvs = pvals_from_Z(Zs)[iu, ju]
        return set(np.where(to_flat(bh_threshold(pvs, ALPHA)))[0])

def mean_pairwise_jaccard(sets):
    js = []
    for i in range(len(sets)):
        for j in range(i+1, len(sets)):
            a, b = sets[i], sets[j]
            u = len(a | b)
            js.append(len(a & b)/u if u else 1.0)
    return float(np.mean(js)) if js else float("nan")

sub_idx1 = [rng.choice(n1, int(FRAC*n1), replace=False) for _ in range(K)]
sub_idx2 = [rng.choice(n2, int(FRAC*n2), replace=False) for _ in range(K)]

for method in ["LCT-B", "BH"]:
    sets = [discoveries_on(X[i1], Y[i2], method)
            for i1, i2 in zip(sub_idx1, sub_idx2)]
    sizes = [len(s) for s in sets]
    print(f"{method:6s}  mean discoveries {np.mean(sizes):8.1f}  "
          f"stability (mean pairwise Jaccard) {mean_pairwise_jaccard(sets):.3f}")

LCT-B   mean discoveries   9676.2  stability (mean pairwise Jaccard) 0.523
BH      mean discoveries  20571.2  stability (mean pairwise Jaccard) 0.808


In [8]:
# Cell 7: NEGATIVE CONTROL (known null) -> direct FDR check on real data
# The 2019-vs-2020 comparison has REAL signal (correlations genuinely shift in a
# crash), so "BH finds more" cannot separate over-rejection from higher power, and
# the Cell-6 stability metric is confounded by set size. Here we split ONE regime
# into two random halves of days: both halves estimate the SAME correlation matrix,
# so the true number of differential edges is ZERO. Every discovery is therefore a
# FALSE positive -- a direct measurement of FDR control on real, heavy-tailed data.
# This is the real-data analog of the null-calibration result in Section 4.

K_NC = 10
rng_nc = np.random.default_rng(RNG_SEED)
data_nc = X                      # calm 2019 regime (set to Y to check the crash regime)
n_nc = data_nc.shape[0]
half = n_nc // 2

def false_discoveries(A, Bmat):
    Ts, RA, RB = lct_edge_stat(A, Bmat, var_method="cai_liu")
    _, m_ln = lct_threshold_normal(Ts, alpha=ALPHA)
    _, m_lb, _ = lct_threshold_bootstrap(A, Bmat, alpha=ALPHA, B=B_BOOT, rng=0)
    Zc  = two_group_z_stat(RA, RB, A.shape[0], Bmat.shape[0])
    pvc = pvals_from_Z(Zc)[iu, ju]
    return (int(to_flat(bh_threshold(pvc, ALPHA)).sum()),
            int(to_flat(by_threshold(pvc, ALPHA)).sum()),
            int(to_flat(m_ln).sum()),
            int(to_flat(m_lb).sum()))

counts = {k: [] for k in ["BH", "BY", "LCT-N", "LCT-B"]}
for _ in range(K_NC):
    perm = rng_nc.permutation(n_nc)
    A, Bmat = data_nc[perm[:half]], data_nc[perm[half:2 * half]]
    bh_c, by_c, ln_c, lb_c = false_discoveries(A, Bmat)
    counts["BH"].append(bh_c); counts["BY"].append(by_c)
    counts["LCT-N"].append(ln_c); counts["LCT-B"].append(lb_c)

print(f"Negative control: two random halves of the SAME regime (true differences = 0)")
print(f"n per half = {half}, edges = {M}, alpha = {ALPHA}, {K_NC} splits")
print(f"Mean FALSE discoveries (should be ~0 if FDR is controlled):")
for k in ["BH", "BY", "LCT-N", "LCT-B"]:
    c = counts[k]
    print(f"  {k:6s}: mean {np.mean(c):8.1f}   max {max(c):6d}   "
          f"implied FDP {np.mean(c)/M:6.3f}")

Negative control: two random halves of the SAME regime (true differences = 0)
n per half = 126, edges = 26796, alpha = 0.05, 10 splits
Mean FALSE discoveries (should be ~0 if FDR is controlled):
  BH    : mean    103.4   max    689   implied FDP  0.004
  BY    : mean      8.1   max     39   implied FDP  0.000
  LCT-N : mean      5.6   max     27   implied FDP  0.000
  LCT-B : mean      0.0   max      0   implied FDP  0.000
